In [1]:
#all necessary imports
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import os

# TA indicators
import ta
from ta.trend import PSARIndicator

# Data Normalization and Scaling
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scikit-learn utilities
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# hyperparameter optimization
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

print("Torch version:", torch.__version__)
print("Polars version:", pl.__version__)
print("TQDM imported successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("TA library imported successfully!")
print("Optuna version:",optuna.__version__)

c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training device: cuda
Torch version: 2.5.1+cu121
Polars version: 0.20.26
TQDM imported successfully!
NumPy version: 1.26.4
Pandas version: 2.2.2
TA library imported successfully!
Optuna version: 4.8.0


In [2]:
#special case company
"""
"4348"

"""

# done training
"""
"4321", "8100"

"""



symbols = ["2120", "6010", "8250"]
#  , "8300", "2200", "1060", "1010", "4190", "2222", "7030", "7010", "2050", "2060"
# "1321", "1180", "2285", "8200", "1050", "1030", "4081", "2286", "1150", 
# "2320", "3080", "4340", "3090", "4342", "2283", "2284", "1323", "4009", 
# "4003", "4334", "2020", "1140", "4084", "7020", "2370", "3030", "4261", 
# "4012", "1020", "1830", "2150", "4260", "3020", "4333", "4001", "1835", 
# "4165", "1320", "4007", "3040", "8280", "4347", "2382", "2223", "4163", 
# "6070", "4262", "4083", "3010", "4164", "2270", "3003", "1834", "1120", 
# "1303", "8170", "1214", "8010", "2081", "4162", "7204", "7202", "2281", 
# "3002", "3050", "1833", "4143", "8030", "4250", "3005", "2230", "4280", 
# "4020", "4004", "8210", "4142", "4263", "2280", "1212", "4002", "8012", 
# "4150", "8160", "1831", "8070", "4051", "1322", "3004", "1182", "6017", 
# "2080", "4161", "4031", "3060", "4071", "1302", "4322", "8020", "4264", 
# "4016", "4072", "4300", "4090", "6001", "6004", "7040", "4191", "4332", 
# "2381", "6014", "4017", "4015", "4324", "6015", "2310", "4336", "4006", 
# "2300", "4338", "8230", "4082", "4144", "4200", "4014", "1111", "4291", 
# "4339", "4013", "2282", "4325", "1304", "7203", "4018", "2084", "7200", 
# "4100", "2070", "2287", "4331", "2190", "2250", "1211", "8313", "4193", 
# "8180", "1183", "2290", "2240", "6020", "4192", "4345", "1301", "8270",
# "4180", "8240", "4210", "8040", "4130", "8310", "4290", "4344", "6016", "4050", 
# "2082", "4040", "1210", "2083", "4080", "6013", "2010", "4170", "3007", "4240"



dfs = {}

for s in symbols:
    filepath = f"{s}_daily.csv"
    
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue
    
    df = pd.read_csv(filepath)
    dfs[s] = df
    print(f"\n── {s} ──────────────────────────")
    print(f"  Shape : {df.shape}")
    print(f"  Head  :\n{df.head()}")
    print(f"  Dtypes:\n{df.dtypes}")
    print(f"  Nulls :\n{df.isnull().sum()}")

print(f"\n── Loaded {len(dfs)}/{len(symbols)} tickers ──")


── 2120 ──────────────────────────
  Shape : (5541, 26)
  Head  :
              datetime  ticker  open_price  high_price  low_price  \
0  2004-02-24 16:00:00    2120    4.201012    4.250827   4.184407   
1  2004-02-25 16:00:00    2120    4.217617    4.267431   4.134593   
2  2004-02-28 16:00:00    2120    4.217617    4.217617   4.184407   
3  2004-02-29 16:00:00    2120    4.201012    4.234222   4.184407   
4  2004-03-01 16:00:00    2120    4.217617    4.267431   4.217617   

   close_price        volume    EMA_10    EMA_20  EMA_ratio  ...  ATR_ratio  \
0     4.217617  2.801601e+05  4.262173  4.274540   0.986683  ...   0.729432   
1     4.184407  1.434375e+06  4.248034  4.265956   0.980884  ...   0.773082   
2     4.201012  8.228047e+05  4.239484  4.259771   0.986206  ...   0.760319   
3     4.217617  1.049848e+06  4.235509  4.255756   0.991038  ...   0.758729   
4     4.234222  1.126091e+06  4.235275  4.253706   0.995420  ...   0.757161   

     BB_pct  realized_vol           OBV  OB

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import pickle



SEQ_LEN = 30

feature_cols = [
    # OHLCV (5)
    'open_price', 'high_price', 'low_price', 'close_price', 'volume',
    # Trend (4)
    'EMA_10', 'EMA_20', 'EMA_ratio', 'MACD_hist',
    # Momentum (4)
    'RSI_14', 'ROC_5', 'ROC_10', 'ROC_20',
    # Volatility (4)
    'ATR_14', 'ATR_ratio', 'BB_pct', 'realized_vol',
    # Volume (5)
    'OBV', 'OBV_momentum', 'volume_ratio', 'volume_surge', 'MFI_14',
    # Lag features (15)
    'close_lag_1', 'close_lag_2', 'close_lag_3', 'close_lag_4', 'close_lag_5',
    'returns_lag_1', 'returns_lag_2', 'returns_lag_3', 'returns_lag_4', 'returns_lag_5',
    'volume_lag_1', 'volume_lag_2', 'volume_lag_3', 'volume_lag_4', 'volume_lag_5',
]

target_cols = ['price_pct_change', 'volume_pct_change_log']

print(f"Total features: {len(feature_cols)}")  # should be 38


def create_sequences(X, y, seq_len):
    sequences, targets = [], []
    for i in range(seq_len, len(X)):
        sequences.append(X[i - seq_len:i])
        targets.append(y[i])
    return np.array(sequences), np.array(targets)


prepared = {}  # stores all data per symbol

for symbol in symbols:
    print(f"\n{'─'*50}")
    print(f"Processing {symbol}...")

    # ── 1. Load ───────────────────────────────────────────────────────────────
    try:
        df = pd.read_csv(f'{symbol}_daily.csv', parse_dates=["datetime"])
    except FileNotFoundError:
        print(f"  WARNING — {symbol}_daily.csv not found, skipping.")
        continue

    df = df.sort_values("datetime").reset_index(drop=True)

    # ── 2. Add lag features ───────────────────────────────────────────────────
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) * 100
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    # ── 3. Fix volume target — log return ─────────────────────────────────────
    df["volume_log"]            = np.log1p(df["volume"])
    df["volume_pct_change_log"] = df["volume_log"].diff(1).shift(-1)
    df = df.drop(columns=["volume_log"], errors="ignore")

    df = df.dropna().reset_index(drop=True)
    print(f"  Rows after lag + dropna: {len(df):,}")

    # ── 4. Outlier removal ────────────────────────────────────────────────────
    outlier_sigma = {'price_pct_change': 3, 'volume_pct_change_log': 2}
    for col, sigma in outlier_sigma.items():
        if col not in df.columns:
            continue
        mean, std = df[col].mean(), df[col].std()
        df = df[df[col].between(mean - sigma * std, mean + sigma * std)]

    df = df.reset_index(drop=True)
    print(f"  Rows after outlier removal: {len(df):,}")

    # ── 5. Percent-based split (70 / 15 / 15) ────────────────────────────────
    n         = len(df)
    train_end = int(n * 0.70)
    val_end   = int(n * 0.85)

    train_df = df.iloc[:train_end]
    val_df   = df.iloc[train_end:val_end]
    test_df  = df.iloc[val_end:]

    print(f"  Train : {len(train_df):,} ({train_df['datetime'].min().date()} → {train_df['datetime'].max().date()})")
    print(f"  Val   : {len(val_df):,} ({val_df['datetime'].min().date()} → {val_df['datetime'].max().date()})")
    print(f"  Test  : {len(test_df):,} ({test_df['datetime'].min().date()} → {test_df['datetime'].max().date()})")

    # ── 6. Check all feature cols exist ──────────────────────────────────────
    missing = [c for c in feature_cols + target_cols if c not in df.columns]
    if missing:
        print(f"  WARNING — missing columns: {missing}, skipping.")
        continue

    # ── 7. Scale ──────────────────────────────────────────────────────────────
    feature_scaler = StandardScaler()
    target_scaler  = StandardScaler()

    train_X = feature_scaler.fit_transform(train_df[feature_cols])
    val_X   = feature_scaler.transform(val_df[feature_cols])
    test_X  = feature_scaler.transform(test_df[feature_cols])

    train_y = target_scaler.fit_transform(train_df[target_cols])
    val_y   = target_scaler.transform(val_df[target_cols])
    test_y  = target_scaler.transform(test_df[target_cols])

    # ── 8. Sliding window sequences ───────────────────────────────────────────
    X_train, y_train = create_sequences(train_X, train_y, SEQ_LEN)
    X_val,   y_val   = create_sequences(val_X,   val_y,   SEQ_LEN)
    X_test,  y_test  = create_sequences(test_X,  test_y,  SEQ_LEN)

    print(f"  X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

    # ── 9. Save scalers ───────────────────────────────────────────────────────
    os.makedirs(f'models/{symbol}', exist_ok=True)

    with open(f'models/{symbol}/{symbol}_feature_scaler.pkl', "wb") as f:
        pickle.dump(feature_scaler, f)
    with open(f'models/{symbol}/{symbol}_target_scaler.pkl', "wb") as f:
        pickle.dump(target_scaler, f)
    with open(f'models/{symbol}/{symbol}_feature_scaler.pkl', "wb") as f:
        pickle.dump(feature_scaler, f)
    with open(f'models/{symbol}/{symbol}_target_scaler.pkl', "wb") as f:
        pickle.dump(target_scaler, f)

    # ── 10. Store for model training ──────────────────────────────────────────
    prepared[symbol] = {
    "X_train": X_train, "y_train": y_train,
    "X_val":   X_val,   "y_val":   y_val,
    "X_test":  X_test,  "y_test":  y_test,
    # raw scaled — for Optuna to re-sequence with different seq_len
    "train_X_scaled": train_X,  "train_y_scaled": train_y,
    "val_X_scaled":   val_X,    "val_y_scaled":   val_y,
    "test_X_scaled":  test_X,   "test_y_scaled":  test_y,   # ← add this line
    }

print(f"\n── Done: {len(prepared)}/{len(symbol)} tickers prepared ──")

Total features: 37

──────────────────────────────────────────────────
Processing 2120...
  Rows after lag + dropna: 5,534
  Rows after outlier removal: 5,119
  Train : 3,583 (2004-03-03 → 2019-11-26)
  Val   : 768 (2019-11-27 → 2023-03-01)
  Test  : 768 (2023-03-02 → 2026-05-07)
  X_train: (3553, 30, 37) | X_val: (738, 30, 37) | X_test: (738, 30, 37)

──────────────────────────────────────────────────
Processing 6010...
  Rows after lag + dropna: 5,534
  Rows after outlier removal: 5,096
  Train : 3,567 (2004-03-06 → 2019-12-17)
  Val   : 764 (2019-12-18 → 2023-03-05)
  Test  : 765 (2023-03-06 → 2026-05-07)
  X_train: (3537, 30, 37) | X_val: (734, 30, 37) | X_test: (735, 30, 37)

──────────────────────────────────────────────────
Processing 8250...
  Rows after lag + dropna: 4,140
  Rows after outlier removal: 3,843
  Train : 2,690 (2009-09-27 → 2021-07-12)
  Val   : 576 (2021-07-13 → 2023-12-03)
  Test  : 577 (2023-12-04 → 2026-05-07)
  X_train: (2660, 30, 37) | X_val: (546, 30, 37) 

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np


# ── Dataset ───────────────────────────────────────────────────────────────────
class TadawulDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 32
loaders    = {}  # loaders["4030"] → {train, val, test}

for symbol, data in prepared.items():
    train_loader = DataLoader(TadawulDataset(data["X_train"], data["y_train"]), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TadawulDataset(data["X_val"],   data["y_val"]),   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(TadawulDataset(data["X_test"],  data["y_test"]),  batch_size=BATCH_SIZE, shuffle=False)

    loaders[symbol] = {
        "train": train_loader,
        "val":   val_loader,
        "test":  test_loader,
    }

    print(f"\n{symbol} — Train: {len(train_loader)} batches | Val: {len(val_loader)} batches | Test: {len(test_loader)} batches")

    # ── Sanity check ──────────────────────────────────────────────────────────
    sample_X, sample_y = next(iter(train_loader))
    print(f"  Sample X: {sample_X.shape} → (batch, seq_len, features)")
    print(f"  Sample y: {sample_y.shape} → (batch, 2)")


2120 — Train: 112 batches | Val: 24 batches | Test: 24 batches
  Sample X: torch.Size([32, 30, 37]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 2]) → (batch, 2)

6010 — Train: 111 batches | Val: 23 batches | Test: 23 batches
  Sample X: torch.Size([32, 30, 37]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 2]) → (batch, 2)

8250 — Train: 84 batches | Val: 18 batches | Test: 18 batches
  Sample X: torch.Size([32, 30, 37]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 2]) → (batch, 2)


In [9]:
class DirectionalHuberLoss(nn.Module):
    def __init__(self, alpha=0.3, delta=1.0):
        super().__init__()
        self.huber = nn.HuberLoss(delta=delta)
        self.alpha = alpha

    def forward(self, preds, targets):
        huber_loss = self.huber(preds, targets)

        # Directional penalty for both price (col 0) and volume (col 1)
        price_dir_penalty  = torch.mean(torch.relu(-preds[:, 0] * targets[:, 0]))
        volume_dir_penalty = torch.mean(torch.relu(-preds[:, 1] * targets[:, 1]))

        # Weight price direction more (matches your 0.6/0.4 val_dir_acc split)
        dir_penalty = (0.6 * price_dir_penalty) + (0.4 * volume_dir_penalty)

        return huber_loss + self.alpha * dir_penalty

In [ ]:
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

best_params_all = {}


# ── BiLSTM + Attention Model (with LayerNorm) ─────────────────────────────────
class StockPctChangeBiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.attention = nn.Linear(hidden_size * 2, 1)
        
        # Changed: BatchNorm1d -> LayerNorm (works with any batch size)
        self.ln = nn.LayerNorm(hidden_size * 2)
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, 2)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = (attn_weights * lstm_out).sum(dim=1)
        
        # Changed: Use LayerNorm instead of BatchNorm
        out = self.ln(context)
        
        out = self.dropout(out)
        out = self.fc(out)
        return out


def create_sequences(X, y, sl):
    sequences, targets = [], []
    for i in range(sl, len(X)):
        sequences.append(X[i - sl:i])
        targets.append(y[i])
    return np.array(sequences), np.array(targets)


# Clean NaN values before Optuna
print("\nCleaning NaN values from all symbols...")
for symbol, data in prepared.items():
    train_X = data["train_X_scaled"]
    train_y = data["train_y_scaled"]
    val_X = data["val_X_scaled"]
    val_y = data["val_y_scaled"]
    
    # Check and remove NaN
    train_nan_mask = ~(np.isnan(train_X).any(axis=1) | np.isnan(train_y).any(axis=1))
    val_nan_mask = ~(np.isnan(val_X).any(axis=1) | np.isnan(val_y).any(axis=1))
    
    before_train = len(train_X)
    before_val = len(val_X)
    
    data["train_X_scaled"] = train_X[train_nan_mask]
    data["train_y_scaled"] = train_y[train_nan_mask]
    data["val_X_scaled"] = val_X[val_nan_mask]
    data["val_y_scaled"] = val_y[val_nan_mask]
    
    after_train = len(data["train_X_scaled"])
    after_val = len(data["val_X_scaled"])
    
    print(f"  {symbol}: Train {before_train}->{after_train}, Val {before_val}->{after_val}")


# Optuna search
for symbol, data in prepared.items():
    print(f"\n{'═'*55}")
    print(f"  Optuna search — {symbol}")
    print(f"{'═'*55}")

    # Use cleaned data
    train_X = data["train_X_scaled"]
    train_y = data["train_y_scaled"]
    val_X = data["val_X_scaled"]
    val_y = data["val_y_scaled"]

    input_size = train_X.shape[1]

    # ── Objective ─────────────────────────────────────────────────────────────
    def objective(trial):
        hidden_size = trial.suggest_categorical("hidden_size", [64, 128])
        num_layers = trial.suggest_int("num_layers", 2, 3)
        dropout = trial.suggest_float("dropout", 0.2, 0.4, step=0.1)
        lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
        batch_size = trial.suggest_categorical("batch_size", [32, 64])
        seq_len = trial.suggest_categorical("seq_len", [20, 30, 40])
        alpha = trial.suggest_float("alpha", 0.1, 0.5, step=0.1)

        X_tr, y_tr = create_sequences(train_X, train_y, seq_len)
        X_vl, y_vl = create_sequences(val_X, val_y, seq_len)

        # Optional: Add drop_last=True for extra safety (not required with LayerNorm)
        train_ld = DataLoader(
            TadawulDataset(X_tr, y_tr), 
            batch_size=batch_size, 
            shuffle=True,
            drop_last=True  # Prevents last incomplete batch
        )
        val_ld = DataLoader(
            TadawulDataset(X_vl, y_vl), 
            batch_size=batch_size, 
            shuffle=False,
            drop_last=False  # Keep all validation samples
        )

        model = StockPctChangeBiLSTMAttention(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout
        ).to(device)

        criterion = DirectionalHuberLoss(alpha=alpha)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

        EPOCHS = 30
        best_val_acc = 0.0
        patience_ctr = 0
        PATIENCE = 7

        for epoch in range(1, EPOCHS + 1):
            model.train()
            for X_batch, y_batch in train_ld:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                optimizer.zero_grad()
                preds = model(X_batch)
                loss = criterion(preds, y_batch)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            model.eval()
            all_preds, all_actuals = [], []
            with torch.no_grad():
                for X_batch, y_batch in val_ld:
                    preds = model(X_batch.to(device)).cpu().numpy()
                    all_preds.append(preds)
                    all_actuals.append(y_batch.numpy())

            # Check if we have validation batches
            if len(all_preds) == 0:
                return 0.0

            val_preds = np.vstack(all_preds)
            val_actuals = np.vstack(all_actuals)

            price_acc = np.mean(np.sign(val_preds[:, 0]) == np.sign(val_actuals[:, 0]))
            volume_acc = np.mean(np.sign(val_preds[:, 1]) == np.sign(val_actuals[:, 1]))
            val_acc = (price_acc * 0.6) + (volume_acc * 0.4)

            trial.report(val_acc, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    break

        return best_val_acc

    # ── Run Optuna ────────────────────────────────────────────────────────────
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
        study_name=f"bilstm_attention_{symbol}"
    )

    study.optimize(objective, n_trials=50, timeout=3600)

    best_params_all[symbol] = study.best_params

    # ── Results ───────────────────────────────────────────────────────────────
    print(f"\n  Best val dir acc : {study.best_value:.4f}")
    print(f"  Best trial       : {study.best_trial.number}")
    print(f"  Best params:")
    for k, v in study.best_params.items():
        print(f"    {k:15s} : {v}")

    print(f"\n  Top 5 Trials:")
    trials_df = study.trials_dataframe().sort_values("value", ascending=False).head()
    print(trials_df[["number", "value", "params_hidden_size",
                      "params_num_layers", "params_dropout",
                      "params_lr", "params_batch_size", "params_seq_len"]])

print(f"\n── Optuna done for {len(best_params_all)} symbols ──")

[I 2026-05-14 13:55:33,882] A new study created in memory with name: bilstm_attention_2120


Using device: cuda

Cleaning NaN values from all symbols...
  2120: Train 3583->3583, Val 768->768
  6010: Train 3567->3567, Val 764->764
  8250: Train 2690->2690, Val 576->576

═══════════════════════════════════════════════════════
  Optuna search — 2120
═══════════════════════════════════════════════════════


[I 2026-05-14 13:55:42,367] Trial 0 finished with value: 0.5355614973262033 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 20, 'alpha': 0.1}. Best is trial 0 with value: 0.5355614973262033.
[I 2026-05-14 13:55:46,617] Trial 1 finished with value: 0.5456043956043956 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.2, 'lr': 0.0002049268011541737, 'batch_size': 64, 'seq_len': 40, 'alpha': 0.1}. Best is trial 1 with value: 0.5456043956043956.
[I 2026-05-14 13:55:51,827] Trial 2 finished with value: 0.5612466124661246 and parameters: {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.00021839352923182988, 'batch_size': 64, 'seq_len': 30, 'alpha': 0.1}. Best is trial 2 with value: 0.5612466124661246.
[I 2026-05-14 13:56:04,931] Trial 3 finished with value: 0.5506775067750678 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.2, 'lr': 0.0001465352103067

In [49]:
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import os
import time

trained_models = {}  # stores best val dir acc per symbol

# ── Directional Accuracy Helper ───────────────────────────────────────────────
def compute_val_dir_acc(model, loader, device):
    model.eval()
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            preds = model(X_batch.to(device)).cpu().numpy()
            all_preds.append(preds)
            all_actuals.append(y_batch.numpy())
    preds   = np.vstack(all_preds)
    actuals = np.vstack(all_actuals)
    price_acc  = np.mean(np.sign(preds[:, 0]) == np.sign(actuals[:, 0]))
    volume_acc = np.mean(np.sign(preds[:, 1]) == np.sign(actuals[:, 1]))
    return (price_acc * 0.6) + (volume_acc * 0.4), price_acc, volume_acc


for symbol, data in prepared.items():
    print(f"\n{'═'*55}")
    print(f"  Training — {symbol}")
    print(f"{'═'*55}")

    best = best_params_all[symbol]
    print(f"  Best params: {best}")

    # ── Rebuild sequences with best seq_len ──────────────────────────────────
    X_train_f, y_train_f = create_sequences(data["train_X_scaled"], data["train_y_scaled"], best["seq_len"])
    X_val_f,   y_val_f   = create_sequences(data["val_X_scaled"],   data["val_y_scaled"],   best["seq_len"])
    X_test_f,  y_test_f  = create_sequences(data["test_X_scaled"],  data["test_y_scaled"],  best["seq_len"])

    train_loader = DataLoader(TadawulDataset(X_train_f, y_train_f), batch_size=best["batch_size"], shuffle=True)
    val_loader   = DataLoader(TadawulDataset(X_val_f,   y_val_f),   batch_size=best["batch_size"], shuffle=False)
    test_loader  = DataLoader(TadawulDataset(X_test_f,  y_test_f),  batch_size=best["batch_size"], shuffle=False)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = best["hidden_size"],
        num_layers  = best["num_layers"],
        dropout     = best["dropout"]
    ).to(device)

    criterion = DirectionalHuberLoss(alpha=best["alpha"])
    optimizer = torch.optim.Adam(model.parameters(), lr=best["lr"], weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    # ── Config ────────────────────────────────────────────────────────────────
    EPOCHS           = 100
    PATIENCE         = 15
    best_val_dir_acc = 0.0
    patience_ctr     = 0
    history          = {"train_loss": [], "val_loss": [], "val_dir_acc": []}

    os.makedirs(f"models/{symbol}", exist_ok=True)
    save_path = f"models/{symbol}/{symbol}_best_bilstm.pt"

    # ── Training Loop ─────────────────────────────────────────────────────────
    print(f"\n{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val DirAcc':>10} | {'Time':>6}")
    print("-" * 58)

    for epoch in range(1, EPOCHS + 1):
        start = time.time()

        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            preds = model(X_batch)
            loss  = criterion(preds, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                preds   = model(X_batch)
                loss    = criterion(preds, y_batch)
                val_losses.append(loss.item())

        train_loss                = np.mean(train_losses)
        val_loss                  = np.mean(val_losses)
        val_dir_acc, p_acc, v_acc = compute_val_dir_acc(model, val_loader, device)
        elapsed                   = time.time() - start

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_dir_acc"].append(val_dir_acc)

        print(f"{epoch:>6} | {train_loss:>10.6f} | {val_loss:>10.6f} | "
              f"{val_dir_acc:>10.4f} | {elapsed:>5.1f}s")

        scheduler.step(val_loss)

        if val_dir_acc > best_val_dir_acc:
            best_val_dir_acc = val_dir_acc
            patience_ctr     = 0
            torch.save(model.state_dict(), save_path)
            print(f"         ✓ Saved  (price_acc={p_acc:.4f}, vol_acc={v_acc:.4f})")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n  Early stopping at epoch {epoch}.")
                break

    trained_models[symbol] = {"best_val_dir_acc": best_val_dir_acc, "history": history}
    print(f"\n  Best val dir acc: {best_val_dir_acc:.4f}")

print(f"\n── Training done for {len(trained_models)} symbols ──")
for s, r in trained_models.items():
    print(f"  {s}: best val dir acc = {r['best_val_dir_acc']:.4f}")


═══════════════════════════════════════════════════════
  Training — 4321
═══════════════════════════════════════════════════════
  Best params: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.001961331382415546, 'batch_size': 64, 'seq_len': 30, 'alpha': 0.1}

 Epoch | Train Loss |   Val Loss | Val DirAcc |   Time
----------------------------------------------------------
     1 |   0.566004 |   0.311452 |     0.4817 |   0.4s
         ✓ Saved  (price_acc=0.4856, vol_acc=0.4760)
     2 |   0.488593 |   0.315591 |     0.4750 |   0.3s
     3 |   0.459615 |   0.327860 |     0.5000 |   0.3s
         ✓ Saved  (price_acc=0.5192, vol_acc=0.4712)
     4 |   0.444159 |   0.322719 |     0.5106 |   0.3s
         ✓ Saved  (price_acc=0.5337, vol_acc=0.4760)
     5 |   0.444141 |   0.313108 |     0.5173 |   0.3s
         ✓ Saved  (price_acc=0.5481, vol_acc=0.4712)
     6 |   0.428018 |   0.312548 |     0.5096 |   0.3s
     7 |   0.423453 |   0.316739 |     0.5010 |   0.4s
     8 |   0.4

DIAGNOSTIC

In [50]:
# ── Evaluation on Test Set ────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("EVALUATION ON TEST SET")
print("=" * 50)

def evaluate(actuals, preds, label):
    mae     = np.mean(np.abs(actuals - preds))
    rmse    = np.sqrt(np.mean((actuals - preds) ** 2))
    dir_acc = np.mean(np.sign(actuals) == np.sign(preds)) * 100
    print(f"\n  [{label}]")
    print(f"  MAE             : {mae:.4f}%")
    print(f"  RMSE            : {rmse:.4f}%")
    print(f"  Directional Acc : {dir_acc:.2f}%")

for symbol in symbols:
    if symbol not in prepared:
        print(f"\n  Skipping {symbol} — not in prepared dict.")
        continue
    if symbol not in trained_models:
        print(f"\n  Skipping {symbol} — no trained model found.")
        continue

    print(f"\n{'═'*55}")
    print(f"  Evaluating — {symbol}")
    print(f"{'═'*55}")

    best = best_params_all[symbol]
    data = prepared[symbol]

    # ── Rebuild test loader with best seq_len ─────────────────────────────────
    X_test_f, y_test_f = create_sequences(
        data["test_X_scaled"], data["test_y_scaled"], best["seq_len"]
    )
    test_loader = DataLoader(
        TadawulDataset(X_test_f, y_test_f),
        batch_size=best["batch_size"],
        shuffle=False
    )

    # ── Load best model ───────────────────────────────────────────────────────
    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = best["hidden_size"],
        num_layers  = best["num_layers"],
        dropout     = best["dropout"]
    ).to(device)

    model.load_state_dict(torch.load(
        f"models/{symbol}/{symbol}_best_bilstm.pt",
        weights_only=True
    ))
    model.eval()

    # ── Load target scaler ────────────────────────────────────────────────────
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "rb") as f:
        target_scaler = pickle.load(f)

    # ── Inference ─────────────────────────────────────────────────────────────
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            preds   = model(X_batch.to(device)).cpu().numpy()
            actuals = y_batch.numpy()
            all_preds.append(preds)
            all_actuals.append(actuals)

    all_preds   = np.vstack(all_preds)
    all_actuals = np.vstack(all_actuals)

    # ── Inverse transform ─────────────────────────────────────────────────────
    all_preds   = target_scaler.inverse_transform(all_preds)
    all_actuals = target_scaler.inverse_transform(all_actuals)

    price_preds_pct    = all_preds[:, 0]
    price_actuals_pct  = all_actuals[:, 0]
    volume_preds_pct   = np.expm1(all_preds[:, 1])   * 100
    volume_actuals_pct = np.expm1(all_actuals[:, 1]) * 100

    # ── Metrics ───────────────────────────────────────────────────────────────
    evaluate(price_actuals_pct,  price_preds_pct,  "Price % Change")
    evaluate(volume_actuals_pct, volume_preds_pct, "Volume % Change")

    # ── Save predictions ──────────────────────────────────────────────────────
    results_df = pd.DataFrame({
        "actual_price_pct"    : price_actuals_pct,
        "predicted_price_pct" : price_preds_pct,
        "actual_volume_pct"   : volume_actuals_pct,
        "predicted_volume_pct": volume_preds_pct,
    })
    out_path = f"models/{symbol}/{symbol}_test_predictions.csv"
    results_df.to_csv(out_path, index=False)
    print(f"\n  Predictions saved: {out_path}")

print(f"\n── Evaluation done for {len(symbols)} symbols ──")


EVALUATION ON TEST SET

═══════════════════════════════════════════════════════
  Evaluating — 4321
═══════════════════════════════════════════════════════

  [Price % Change]
  MAE             : 1.0047%
  RMSE            : 1.2774%
  Directional Acc : 50.96%

  [Volume % Change]
  MAE             : 45.1788%
  RMSE            : 61.9420%
  Directional Acc : 50.48%

  Predictions saved: models/4321/4321_test_predictions.csv

═══════════════════════════════════════════════════════
  Evaluating — 8100
═══════════════════════════════════════════════════════

  [Price % Change]
  MAE             : 1.5004%
  RMSE            : 2.0126%
  Directional Acc : 52.96%

  [Volume % Change]
  MAE             : 51.8699%
  RMSE            : 70.8650%
  Directional Acc : 52.96%

  Predictions saved: models/8100/8100_test_predictions.csv

── Evaluation done for 2 symbols ──


In [51]:
import pandas as pd
import numpy as np

for symbol in symbols:
    csv_path = f"models/{symbol}/{symbol}_test_predictions.csv"

    try:
        results = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"\n  Skipping {symbol} — {csv_path} not found.")
        continue

    price_actual    = results["actual_price_pct"]
    price_predicted = results["predicted_price_pct"]

    correct   = (np.sign(price_actual) == np.sign(price_predicted))
    up_mask   = price_actual > 0
    down_mask = price_actual < 0

    print(f"\n{'═'*55}")
    print(f"  {symbol} — Honest Model Baseline")
    print(f"{'═'*55}")
    print(f"  Actual    mean : {price_actual.mean():.4f}   std: {price_actual.std():.4f}")
    print(f"  Predicted mean : {price_predicted.mean():.4f}   std: {price_predicted.std():.4f}")
    print(f"\n  Overall Dir Acc : {correct.mean()*100:.2f}%")
    print(f"  When actual UP  : {correct[up_mask].mean()*100:.2f}%  ({up_mask.sum()} samples)")
    print(f"  When actual DOWN: {correct[down_mask].mean()*100:.2f}%  ({down_mask.sum()} samples)")
    print(f"\n  % predicted UP  : {(price_predicted > 0).mean()*100:.2f}%")
    print(f"  % actual UP     : {(price_actual > 0).mean()*100:.2f}%")

    print(f"\n  Dir Acc by Move Size:")
    bins   = [0, 0.5, 1.0, 2.0, 5.0, 100.0]
    labels = ["<0.5%", "0.5-1%", "1-2%", "2-5%", ">5%"]
    price_actual_abs = price_actual.abs()
    for i, label in enumerate(labels):
        mask = (price_actual_abs >= bins[i]) & (price_actual_abs < bins[i+1])
        if mask.sum() > 0:
            acc = correct[mask].mean() * 100
            print(f"    {label:>8} moves : {acc:.2f}%  ({mask.sum()} samples)")

print(f"\n── Diagnostics done for {len(symbols)} symbols ──")


═══════════════════════════════════════════════════════
  4321 — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : -0.0949   std: 1.2825
  Predicted mean : -0.0359   std: 0.1089

  Overall Dir Acc : 50.96%
  When actual UP  : 55.21%  (96 samples)
  When actual DOWN: 47.32%  (112 samples)

  % predicted UP  : 53.85%
  % actual UP     : 46.15%

  Dir Acc by Move Size:
       <0.5% moves : 51.79%  (56 samples)
      0.5-1% moves : 54.41%  (68 samples)
        1-2% moves : 50.00%  (64 samples)
        2-5% moves : 40.00%  (20 samples)

═══════════════════════════════════════════════════════
  8100 — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : -0.0255   std: 2.0063
  Predicted mean : -0.0411   std: 0.0930

  Overall Dir Acc : 52.96%
  When actual UP  : 24.37%  (279 samples)
  When actual DOWN: 77.20%  (329 samples)

  % predicted UP  : 23.52%
  % actual UP     : 45.89%

  Dir Acc by Move Siz